## **Iterative Imputation (MICE)**

Topic Roadmap

**1. Imports and Setup**

**2. Data Preparation & Injection of Missing Values**

**3. Under the Hood: Manual MICE Implementation**
- 3.1 Iteration 0 (Mean Imputation)
- 3.2 Iteration 1 (Round-Robin Regression)
- 3.3 Iteration 2 & Convergence Check

**4. Production Implementation: Scikit-Learn `IterativeImputer`**
**5. Key Revision Notes**

### **1. Imports and Setup**

Import standard libraries and the specific algorithms required. 

**Note:** To use Scikit-Learn's `IterativeImputer`, you must explicitly enable it via `sklearn.experimental` because it is still technically considered an experimental feature.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

# Scikit-Learn MICE Imports
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

### **2. Data Preparation & Injection of Missing Values**

Load the dataset, take a small sample to understand the mechanics, and artificially inject missing values (`NaN`) to simulate incomplete data.

In [ ]:
# Load and scale down dataset for readability

df_raw = pd.read_csv('docs/Lecture-027-50_Startups.csv')[['R&D Spend', 'Administration', 'Marketing Spend', 'Profit']]
df_scaled = np.round(df_raw / 10000)

# Sample 5 rows and isolate features (excluding target 'Profit')
np.random.seed(9)
df = df_scaled.sample(5).iloc[:, 0:-1].copy()

df

,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,4.0,5.0,20.0
2,15.0,10.0,41.0
14,12.0,16.0,26.0
44,2.0,15.0,3.0


In [4]:
# Inject Missing Values diagonally for demonstration
df.iloc[1, 0] = np.nan  # Missing R&D Spend
df.iloc[3, 1] = np.nan  # Missing Administration
df.iloc[-1, -1] = np.nan # Missing Marketing Spend

df

,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,NaN,5.0,20.0
2,15.0,10.0,41.0
14,12.0,NaN,26.0
44,2.0,15.0,NaN


### **3. Under the Hood: Manual MICE Implementation**

Multivariate Imputation by Chained Equations (MICE) models each feature with missing values as a function of other features in a round-robin fashion.

**3.1 Iteration 0 (Mean Imputation)**

The algorithm initializes by filling all missing values with the mean of their respective columns. This provides a baseline complete dataset to perform regressions.

In [5]:
# Base initialization: fill with mean
df0 = pd.DataFrame()
df0['R&D Spend'] = df['R&D Spend'].fillna(df['R&D Spend'].mean())
df0['Administration'] = df['Administration'].fillna(df['Administration'].mean())
df0['Marketing Spend'] = df['Marketing Spend'].fillna(df['Marketing Spend'].mean())

df0

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,9.25,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.25,26.00
44,2.00,15.00,29.25


**3.2 Iteration 1 (Round-Robin Regression)**

We treat each column with missing data as a target variable (`y`) and the other columns as features (`X`). We predict the missing value and update the dataset.

In [6]:
# Setup Iteration 1 DataFrame
df1 = df0.copy()

lr = LinearRegression()

**Step 1: Predict missing `R&D Spend`**

In [8]:
# Re-introduce NaN for the value we are about to predict
df1.iloc[1, 0] = np.nan

# X: Administration and Marketing Spend | y: R&D Spend (excluding row 1)
X_train = df1.iloc[[0, 2, 3, 4], 1:3]
y_train = df1.iloc[[0, 2, 3, 4], 0]

# Train and predict
lr.fit(X_train, y_train)
pred_rd = lr.predict(df1.iloc[1, 1:].values.reshape(1, 2))

# Update DataFrame with prediction
df1.iloc[1, 0] = pred_rd[0]
df1

C:\Users\ART\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


,R&D Spend,Administration,Marketing Spend
21,8.000000,15.00,30.00
37,23.141587,5.00,20.00
2,15.000000,10.00,41.00
14,12.000000,11.25,26.00
44,2.000000,15.00,29.25


**Step 2: Predict missing `Administration`**

In [10]:
# Re-introduce NaN for the target value
df1.iloc[3, 1] = np.nan

# X: R&D Spend and Marketing Spend | y: Administration (excluding row 3)
X_train = df1.iloc[[0, 1, 2, 4], [0, 2]]
y_train = df1.iloc[[0, 1, 2, 4], 1]

lr.fit(X_train, y_train)
pred_admin = lr.predict(df1.iloc[3, [0, 2]].values.reshape(1, 2))

df1.iloc[3, 1] = pred_admin[0]
df1

C:\Users\ART\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


,R&D Spend,Administration,Marketing Spend
21,8.000000,15.000000,30.00
37,23.141587,5.000000,20.00
2,15.000000,10.000000,41.00
14,12.000000,11.063618,26.00
44,2.000000,15.000000,29.25


**Step 3: Predict missing `Marketing Spend`**

In [12]:
# Re-introduce NaN for the target value
df1.iloc[4, -1] = np.nan

# X: R&D Spend and Administration | y: Marketing Spend (excluding row 4)
X_train = df1.iloc[0:4, 0:2]
y_train = df1.iloc[0:4, -1]

lr.fit(X_train, y_train)
pred_marketing = lr.predict(df1.iloc[4, 0:2].values.reshape(1, 2))

df1.iloc[4, -1] = pred_marketing[0]
df1

C:\Users\ART\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


,R&D Spend,Administration,Marketing Spend
21,8.000000,15.000000,30.000000
37,23.141587,5.000000,20.000000
2,15.000000,10.000000,41.000000
14,12.000000,11.063618,26.000000
44,2.000000,15.000000,31.601847


**3.3 Iteration 2 & Convergence Check**

The algorithm repeats this round-robin prediction process until the difference between the current iteration and the previous iteration approaches zero (convergence).

In [13]:
# Setup Iteration 2 DataFrame
df2 = df1.copy()

# Predict R&D Spend
df2.iloc[1, 0] = np.nan
lr.fit(df2.iloc[[0, 2, 3, 4], 1:3], df2.iloc[[0, 2, 3, 4], 0])
df2.iloc[1, 0] = lr.predict(df2.iloc[1, 1:].values.reshape(1, 2))[0]

# Predict Administration
df2.iloc[3, 1] = np.nan
lr.fit(df2.iloc[[0, 1, 2, 4], [0, 2]], df2.iloc[[0, 1, 2, 4], 1])
df2.iloc[3, 1] = lr.predict(df2.iloc[3, [0, 2]].values.reshape(1, 2))[0]

# Predict Marketing Spend
df2.iloc[4, -1] = np.nan
lr.fit(df2.iloc[0:4, 0:2], df2.iloc[0:4, -1])
df2.iloc[4, -1] = lr.predict(df2.iloc[4, 0:2].values.reshape(1, 2))[0]

# Calculate difference between Iteration 2 and Iteration 1
difference = df2 - df1
difference

C:\Users\ART\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
C:\Users\ART\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
C:\Users\ART\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


,R&D Spend,Administration,Marketing Spend
21,0.000000,0.000000,0.000000
37,0.687117,0.000000,0.000000
2,0.000000,0.000000,0.000000
14,0.000000,0.167119,0.000000
44,0.000000,0.000000,7.788589


### **4. Production Implementation: Scikit-Learn `IterativeImputer`**

In a production environment, you do not write manual chained equations. Scikit-Learn provides `IterativeImputer` to handle this entire process automatically, including tracking convergence and mapping features.

In [14]:
# Display original dataframe with NaN values
print("Original Data with NaNs:")
display(df)

# Initialize IterativeImputer
# You can specify the estimator (e.g., Random Forest) but it defaults to BayesianRidge/Ridge regression.
mice_imputer = IterativeImputer(estimator=LinearRegression(), max_iter=10, random_state=42)

# Fit and Transform the DataFrame
df_imputed_array = mice_imputer.fit_transform(df)

# Convert back to Pandas DataFrame for readability
df_imputed = pd.DataFrame(df_imputed_array, columns=df.columns, index=df.index)

print("\nData after Scikit-Learn IterativeImputer:")
display(df_imputed)

Original Data with NaNs:


,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,NaN,5.0,20.0
2,15.0,10.0,41.0
14,12.0,NaN,26.0
44,2.0,15.0,NaN



Data after Scikit-Learn IterativeImputer:


,R&D Spend,Administration,Marketing Spend
21,8.000000,15.000000,30.000000
37,26.718225,5.000000,20.000000
2,15.000000,10.000000,41.000000
14,12.000000,13.022458,26.000000
44,2.000000,15.000000,70.692637


### **Key Revision Notes**

- **What it is:** MICE (Multivariate Imputation by Chained Equations) is an imputation strategy that models each feature with missing values as a function of the other features.
- **How it works:** 
  1. Fills missing values with a baseline statistic (like mean) temporarily.
  2. Reverts one column's imputed values back to missing.
  3. Regresses that column on all other columns to predict its missing values.
  4. Cycles through all columns (Chained Equations) until the values converge (stop changing significantly between iterations).
- **Scikit-Learn Implementation:** Handled via `IterativeImputer`. It must be explicitly enabled using `from sklearn.experimental import enable_iterative_imputer`.
- **Advantages:** Highly robust and preserves statistical relationships (variance and covariance) better than univariate methods like simple mean/median imputation.
- **Custom Estimators:** While `IterativeImputer` uses a form of Ridge regression by default, you can pass other algorithms via the `estimator` argument (e.g., `estimator=RandomForestRegressor()`) to capture non-linear relationships.